# 05 — Topological Map

Visualize the vault DAG as a topological elevation map:
- Hierarchy levels (depth in the DAG)
- Node connectivity (in-degree, out-degree)
- GRU prediction attractor analysis
- Why certain nodes become "mountain peaks"

In [1]:
import { DenoVaultReader } from "../src/core/io.ts";
import { parseVault } from "../src/core/parser.ts";
import { buildGraph, topologicalSort } from "../src/core/graph.ts";
import { openVaultStore } from "../src/db/index.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;

const reader = new DenoVaultReader();
const notes = await parseVault(reader, VAULT_PATH);
const graph = buildGraph(notes);
const order = topologicalSort(graph);
let db: any = null;
let allNotes: any[] = [];
try {
  db = await openVaultStore(DB_PATH);
  allNotes = await db.getAllNotes();
} catch (err) {
  console.warn(`[warn] KV store unavailable in this notebook kernel: ${(err as Error).message}`);
  console.warn("[warn] Install/restart a Deno kernel with --unstable-kv to enable GRU and trace charts.");
}

console.log(`Vault: ${notes.length} notes, topo order: ${order.join(' → ')}`);

Vault: 66 notes, topo order: CRM - Contacts → CRM - Deals → CRM - Follow-up Engine → CRM - Pipeline → CRM - Daily Command Board → CRM - Workspace README → REV - Accounts → REV - Account Tier Weight → REV - CSM Owners → REV - Contracts → REV - Account Contract Links → REV - Contract Status Filter → REV - Days To Renewal → REV - Entity Types → REV - Hard Reset Log → REV - Invoices → REV - Invoice Overdue Amount → REV - NPS Records → REV - NPS Freshness → REV - Overdue Invoice Count → REV - Owner Join → REV - Relation Types → REV - Renewal Window Band → REV - Tickets → REV - Account Ticket Links → REV - High Severity Ticket Count → REV - Open Ticket Aging Score → REV - Open Ticket Count → REV - Usage Snapshots → REV - Account Health Inputs → REV - Revenue Risk Score → REV - Churn Pressure Score → REV - Executive Risk Brief → REV - Expansion Readiness Score → REV - Expansion Delta → REV - Expansion Pipeline → REV - Leadership Brief Draft → REV - Owner Priority Matrix → REV - Owner Capacity

In [2]:
// Compute DAG metrics for each node
const noteSet = new Set(notes.map(n => n.name));

// Out-degree: number of dependencies (wikilinks to other notes)
const outDegree = new Map<string, number>();
for (const n of notes) {
  outDegree.set(n.name, n.wikilinks.filter(w => noteSet.has(w)).length);
}

// In-degree: how many notes depend on this one
const inDegree = new Map<string, number>();
for (const name of noteSet) inDegree.set(name, 0);
for (const n of notes) {
  for (const dep of n.wikilinks) {
    if (noteSet.has(dep)) inDegree.set(dep, (inDegree.get(dep) ?? 0) + 1);
  }
}

// Level: longest path from any root to this node
const levels = new Map<string, number>();
function getLevel(name: string): number {
  if (levels.has(name)) return levels.get(name)!;
  const note = notes.find(n => n.name === name);
  if (!note || note.wikilinks.length === 0) { levels.set(name, 0); return 0; }
  const depLevels = note.wikilinks.filter(w => noteSet.has(w)).map(w => getLevel(w));
  const level = depLevels.length > 0 ? Math.max(...depLevels) + 1 : 0;
  levels.set(name, level);
  return level;
}
notes.forEach(n => getLevel(n.name));

// Transitive reach: how many unique notes are in my full subgraph
function transitiveReach(name: string, visited = new Set<string>()): Set<string> {
  if (visited.has(name)) return visited;
  visited.add(name);
  const note = notes.find(n => n.name === name);
  if (note) {
    for (const dep of note.wikilinks) {
      if (noteSet.has(dep)) transitiveReach(dep, visited);
    }
  }
  return visited;
}

// Reverse transitive reach: how many notes can reach me
const reverseEdges = new Map<string, string[]>();
for (const name of noteSet) reverseEdges.set(name, []);
for (const n of notes) {
  for (const dep of n.wikilinks) {
    if (noteSet.has(dep)) reverseEdges.get(dep)!.push(n.name);
  }
}

function reverseReach(name: string, visited = new Set<string>()): Set<string> {
  if (visited.has(name)) return visited;
  visited.add(name);
  for (const parent of (reverseEdges.get(name) ?? [])) {
    reverseReach(parent, visited);
  }
  return visited;
}

console.log('\n  Node Topology:');
console.log('  ' + 'Name'.padEnd(22) + 'Level  In  Out  Reach↓  Reach↑  Type');
console.log('  ' + '─'.repeat(70));

const nodeData: Array<{name: string, level: number, inDeg: number, outDeg: number, reach: number, reverseR: number, type: string}> = [];

for (const name of order) {
  const note = notes.find(n => n.name === name)!;
  const type = note.frontmatter.code ? 'code' : 'value';
  const lvl = levels.get(name) ?? 0;
  const iDeg = inDegree.get(name) ?? 0;
  const oDeg = outDegree.get(name) ?? 0;
  const reach = transitiveReach(name).size;
  const revR = reverseReach(name).size;
  
  nodeData.push({ name, level: lvl, inDeg: iDeg, outDeg: oDeg, reach, reverseR: revR, type });
  console.log(`  ${name.padEnd(22)} L${lvl}     ${iDeg}    ${oDeg}     ${reach}       ${revR}       ${type}`);
}


  Node Topology:


  Name                  Level  In  Out  Reach↓  Reach↑  Type


  ──────────────────────────────────────────────────────────────────────


  CRM - Contacts         L0     1    0     1       2       value


  CRM - Deals            L0     2    0     1       4       value


...

  REV - Dispatch Dry Run L11     1    3     46       2       code


  REV - Dispatch Audit Log L12     0    1     47       1       code


  SHARED - Segment Owner Routing L0     1    0     1       2       value


  CRM - Inbox-to-CRM     L1     0    2     3       1       code


  SHARED - Vault Workspace README L0     0    0     1       1       value


In [3]:
// Topological elevation map: scatter plot with level as Y, position as X
// Size = transitive reach (subgraph size), Color = reverse reach (how many depend on me)

const scatterData = nodeData.map((n, i) => ({
  name: n.name,
  level: n.level,
  reach: n.reach,
  reverseReach: n.reverseR,
  inDegree: n.inDeg,
  outDegree: n.outDeg,
  type: n.type,
  // Elevation score: combined metric of how "peak" this node is
  elevation: n.level + n.reach * 0.5 + n.outDeg * 0.3,
}));

const elevationSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Topological Elevation Map",
  "width": 500,
  "height": 350,
  "data": { "values": scatterData },
  "mark": { "type": "circle", "opacity": 0.8 },
  "encoding": {
    "x": { "field": "reverseReach", "type": "quantitative", "title": "Reverse Reach (how many notes can reach me)" },
    "y": { "field": "level", "type": "quantitative", "title": "DAG Level (depth)" },
    "size": { "field": "reach", "type": "quantitative", "title": "Transitive Reach (subgraph size)", "scale": { "range": [100, 1500] } },
    "color": { "field": "elevation", "type": "quantitative", "scale": { "scheme": "magma" }, "title": "Elevation Score" },
    "tooltip": [
      { "field": "name" },
      { "field": "level" },
      { "field": "reach", "title": "Subgraph Size" },
      { "field": "reverseReach", "title": "Depended On By" },
      { "field": "inDegree" },
      { "field": "outDegree" },
      { "field": "elevation", "format": ".1f" }
    ]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": elevationSpec }, { raw: true });

In [4]:
// DAG as a layered node-link diagram
// Nodes at each level, edges showing dependencies

const edges: Array<{source: string, target: string, sourceLevel: number, targetLevel: number}> = [];
for (const note of notes) {
  for (const dep of note.wikilinks) {
    if (noteSet.has(dep)) {
      edges.push({
        source: dep,
        target: note.name,
        sourceLevel: levels.get(dep) ?? 0,
        targetLevel: levels.get(note.name) ?? 0,
      });
    }
  }
}

// Assign x-positions within each level
const byLevel = new Map<number, string[]>();
for (const [name, lvl] of levels) {
  if (!byLevel.has(lvl)) byLevel.set(lvl, []);
  byLevel.get(lvl)!.push(name);
}

const positions = new Map<string, {x: number, y: number}>();
for (const [lvl, names] of byLevel) {
  names.forEach((name, i) => {
    positions.set(name, {
      x: (i + 1) / (names.length + 1) * 100,
      y: lvl * 100,
    });
  });
}

console.log('\n  DAG Hierarchy (levels):');
const maxLevel = Math.max(...levels.values());
for (let lvl = maxLevel; lvl >= 0; lvl--) {
  const names = byLevel.get(lvl) ?? [];
  const indent = '  '.repeat(lvl);
  console.log(`  L${lvl}: ${indent}${names.join('  |  ')}`);
}

console.log('\n  Edges:');
for (const e of edges) {
  console.log(`  ${e.source} (L${e.sourceLevel}) ──→ ${e.target} (L${e.targetLevel})`);
}


  DAG Hierarchy (levels):


  L12:                         REV - Dispatch Audit Log


  L11:                       REV - Dispatch Dry Run


  L10:                     REV - Dispatch Batch  |  REV - Payload Validator  |  REV - Idempotency Key Builder


  L9:                   REV - Action Dispatch Queue


...

  REV - Accounts (L0) ──→ REV - Task Close Payload (L8)


  REV - Task Upsert Payload (L7) ──→ REV - Task Reassign Payload (L8)


  REV - Owner Workload Balance (L7) ──→ REV - Task Reassign Payload (L8)


  REV - CSM Daily Plan (L6) ──→ REV - Task Upsert Payload (L7)


  REV - Account Contract Links (L1) ──→ REV - Task Upsert Payload (L7)


In [5]:
// Node-link diagram with Vega-Lite
const nodePoints = [...positions.entries()].map(([name, pos]) => ({
  name,
  x: pos.x,
  y: pos.y,
  level: levels.get(name) ?? 0,
  reach: transitiveReach(name).size,
  type: notes.find(n => n.name === name)?.frontmatter.code ? 'code' : 'value',
}));

const edgeLines: Array<{x: number, y: number, x2: number, y2: number}> = edges.map(e => ({
  x: positions.get(e.source)!.x,
  y: positions.get(e.source)!.y,
  x2: positions.get(e.target)!.x,
  y2: positions.get(e.target)!.y,
}));

const dagSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "DAG Hierarchy",
  "width": 600,
  "height": 400,
  "layer": [
    {
      "data": { "values": edgeLines },
      "mark": { "type": "rule", "color": "#666", "opacity": 0.5 },
      "encoding": {
        "x": { "field": "x", "type": "quantitative", "scale": { "domain": [0, 100] }, "axis": null },
        "y": { "field": "y", "type": "quantitative", "scale": { "domain": [-20, (maxLevel + 1) * 100] }, "axis": { "title": "DAG Level" } },
        "x2": { "field": "x2" },
        "y2": { "field": "y2" }
      }
    },
    {
      "data": { "values": nodePoints },
      "mark": { "type": "circle", "opacity": 0.9 },
      "encoding": {
        "x": { "field": "x", "type": "quantitative", "axis": null },
        "y": { "field": "y", "type": "quantitative" },
        "size": { "field": "reach", "type": "quantitative", "scale": { "range": [200, 1200] }, "title": "Subgraph Size" },
        "color": { "field": "type", "type": "nominal", "scale": { "domain": ["value", "code"], "range": ["#4CAF50", "#2196F3"] } },
        "tooltip": [{ "field": "name" }, { "field": "level" }, { "field": "reach" }, { "field": "type" }]
      }
    },
    {
      "data": { "values": nodePoints },
      "mark": { "type": "text", "dy": -15, "fontSize": 11, "fontWeight": "bold" },
      "encoding": {
        "x": { "field": "x", "type": "quantitative" },
        "y": { "field": "y", "type": "quantitative" },
        "text": { "field": "name", "type": "nominal" }
      }
    }
  ]
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": dagSpec }, { raw: true });

In [6]:
// Attractor analysis: which nodes does the GRU converge to and why?
const { GRUInference } = await import("../src/gru/inference.ts");
const { deserializeWeights } = await import("../src/gru/weights.ts");

if (!db) {
  console.log("KV store unavailable in kernel: skipping GRU attractor analysis.");
} else {
const weightsRow = await db.getLatestWeights();
if (weightsRow) {
  const { weights, vocab, config } = await deserializeWeights(weightsRow.blob);
  const gru = new GRUInference(weights, vocab, config);
  
  // For each note: feed its GNN embedding as intent → what does GRU predict?
  console.log('\n  GRU Attractor Analysis:');
  console.log('  (Feed each note\'s embedding as intent → see where GRU routes)');
  console.log('  ' + '─'.repeat(60));
  
  const predictions = new Map<string, number>();
  
  for (const note of allNotes) {
    const emb = note.gnnEmbedding ?? note.embedding;
    if (!emb) continue;
    const pred = gru.predictNext(emb, []);
    predictions.set(pred.name, (predictions.get(pred.name) ?? 0) + 1);
    
    const isAttractor = pred.name === note.name;
    const elevData = nodeData.find(n => n.name === pred.name);
    console.log(`  ${note.name.padEnd(22)} → ${pred.name.padEnd(22)} (L${elevData?.level ?? '?'}, reach=${elevData?.reach ?? '?'}) ${isAttractor ? '⟲ self' : ''}`);
  }
  
  console.log('\n  Attractor frequency:');
  [...predictions.entries()].sort((a,b) => b[1] - a[1]).forEach(([name, count]) => {
    const nd = nodeData.find(n => n.name === name);
    console.log(`  ${count}/${allNotes.length} → ${name} (L${nd?.level}, reach=${nd?.reach}, reverseReach=${nd?.reverseR})`);
  });
  
  console.log('\n  ⚠ Hypothesis: The attractor is the node with highest (level + reach).');
  console.log('  With only', noteSet.size, 'notes, the GRU cannot discriminate — it converges to the topological peak.');
} else {
  console.log('No GRU weights found.');
}
}


  GRU Attractor Analysis:


  (Feed each note's embedding as intent → see where GRU routes)


  ────────────────────────────────────────────────────────────


  Account Contract Links → Upsell Candidate Queue (L?, reach=?) 


  Account Health Inputs  → Upsell Candidate Queue (L?, reach=?) 


...

  81/124 → Upsell Candidate Queue (Lundefined, reach=undefined, reverseReach=undefined)


  41/124 → REV - Dispatch Audit Log (L12, reach=47, reverseReach=1)


  2/124 → REV - Dispatch Dry Run (L11, reach=46, reverseReach=2)



  ⚠ Hypothesis: The attractor is the node with highest (level + reach).


  With only 66 notes, the GRU cannot discriminate — it converges to the topological peak.


In [7]:
// Trace target distribution overlaid with topological metrics
if (!db) {
  console.log("KV store unavailable in kernel: skipping trace distribution chart.");
} else {
const traces = await db.getAllTraces();
const traceCounts = new Map<string, {synthetic: number, real: number}>();
for (const t of traces) {
  if (!traceCounts.has(t.targetNote)) traceCounts.set(t.targetNote, {synthetic: 0, real: 0});
  if (t.synthetic) traceCounts.get(t.targetNote)!.synthetic++;
  else traceCounts.get(t.targetNote)!.real++;
}

const traceBarData: Array<{name: string, count: number, source: string, level: number}> = [];
for (const [name, counts] of traceCounts) {
  traceBarData.push({ name, count: counts.synthetic, source: 'synthetic', level: levels.get(name) ?? 0 });
  traceBarData.push({ name, count: counts.real, source: 'real', level: levels.get(name) ?? 0 });
}

const traceSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Trace Targets vs DAG Level",
  "width": 500,
  "height": 300,
  "data": { "values": traceBarData },
  "mark": "bar",
  "encoding": {
    "x": { "field": "name", "type": "nominal", "sort": { "field": "level" }, "title": "Target Note (sorted by level)" },
    "y": { "field": "count", "type": "quantitative", "stack": true, "title": "Trace Count" },
    "color": { "field": "source", "type": "nominal", "scale": { "range": ["#90CAF9", "#FF7043"] } },
    "tooltip": [{ "field": "name" }, { "field": "source" }, { "field": "count" }, { "field": "level" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": traceSpec }, { raw: true });
}

In [8]:
if (db) db.close();
console.log('\nConclusion:');
console.log('The attractor phenomenon is structural, not a GRU bug.');
console.log('With', noteSet.size, 'notes, the highest-elevation node absorbs all predictions.');
console.log('Remedy: larger vault with more diverse paths, or supervised traces.');


Conclusion:


The attractor phenomenon is structural, not a GRU bug.


With 66 notes, the highest-elevation node absorbs all predictions.


Remedy: larger vault with more diverse paths, or supervised traces.
